# Preprocess masks generated with HistoKit

This notebook includes the code to preprocess annotated regions from artifact masks generated with HistoKit.

## TCGA - Compass NMD Dataset

Annotations for the TCGA - Compass NMD dataset are available for regions on the WSI, so there is a need to crop the masks to the annotated regions.

Masks are cropped using the coordinates saved in the file `coords.csv` and saved in a new folder. The cropped masks are also saved in color format for visualization purposes.

In [1]:
from PIL import Image
import numpy as np
from histokit.savers import HDF5Saver
import pandas as pd
import os
import shutil
from tqdm import tqdm
from histokit.file_utils.file_check import check_gt_pred_folders
import os
Image.MAX_IMAGE_PIXELS = None

classes = {
    "Tissue": [128, 128, 128],
    "Background": [0, 0, 0],
    "Fold": [255, 99, 71],
    "Dark.Spot": [0, 255, 0],
    "Pen": [255, 0, 0],
    "Edge": [255, 0, 255],
    "Out.Of.Focus": [75, 0, 130],
}



/home/jmerta/miniconda3/envs/HistoKit_3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/TCGA_CompassNMD/10x/annotated_coords.csv')
df.head()
svs_path = df["svs_path"].values
svs_path = [os.path.basename(s).split(".svs")[0] for s in svs_path]

df["svs_name"] = svs_path
df.head()

,svs_path,patch_path,matched_path,target_mag,objective_mag,target_downsample,matched_level,level_downsample,matched_level_mag,resize_scale,...,y_10x,patch_width_10x,patch_height_10x,x0,y0,width0,height0,x0_end,y0_end,svs_name
0,/mnt/warehouse/Projects/HE/Data/Artifacts Segm...,/mnt/warehouse/Projects/HE/Data/Artifacts Segm...,/mnt/warehouse/Projects/HE/Data/Artifacts Segm...,10,20.0,2.0,0,1.0,20.0,0.5,...,0,8963,5954,0,0,17926,11908,17926,11908,TCGA-CV-7099-01A-02-BS2.1e152adb-e0cb-4962-800...
1,/mnt/warehouse/Projects/HE/Data/Artifacts Segm...,/mnt/warehouse/Projects/HE/Data/Artifacts Segm...,/mnt/warehouse/Projects/HE/Data/Artifacts Segm...,10,50.0,5.0,1,4.0,12.5,0.8,...,5601,3712,2768,33605,28005,18560,13840,52165,41845,2-12_he_1
2,/mnt/warehouse/Projects/HE/Data/Artifacts Segm...,/mnt/warehouse/Projects/HE/Data/Artifacts Segm...,/mnt/warehouse/Projects/HE/Data/Artifacts Segm...,10,50.0,5.0,1,4.0,12.5,0.8,...,592,6416,11392,24085,2960,32080,56960,56165,59920,33-00_he
3,/mnt/warehouse/Projects/HE/Data/Artifacts Segm...,/mnt/warehouse/Projects/HE/Data/Artifacts Segm...,/mnt/warehouse/Projects/HE/Data/Artifacts Segm...,10,20.0,2.0,0,1.0,20.0,0.5,...,4546,6482,6619,10872,9092,12964,13238,23836,22330,TCGA-08-0520-01Z-00-DX1.2d50b1bc-c1d8-41a4-b42...
4,/mnt/warehouse/Projects/HE/Data/Artifacts Segm...,/mnt/warehouse/Projects/HE/Data/Artifacts Segm...,/mnt/warehouse/Projects/HE/Data/Artifacts Segm...,10,50.0,5.0,1,4.0,12.5,0.8,...,0,5249,6181,25150,0,26245,30905,51395,30905,11-22_he


In [2]:
import os
import re


def parse_grid_search_params(path):
    folder_name = None

    for part in path.split(os.sep):
        if part.startswith("blending_mode_"):
            folder_name = part
            break

    if folder_name is None:
        raise ValueError(f"Cannot find grid-search folder in path: {path}")

    result = {
        "mode_overlap": "",
        "overlap": "",
        "sigma": "",
    }

    mode_match = re.search(r"blending_mode_([^_]+)", folder_name)
    overlap_match = re.search(r"overlap_([0-9]+p[0-9]+|[0-9]+)", folder_name)
    sigma_match = re.search(r"blending_sigma_([0-9]+p[0-9]+|[0-9]+)", folder_name)

    if mode_match:
        result["mode_overlap"] = mode_match.group(1)

    if overlap_match:
        result["overlap"] = float(overlap_match.group(1).replace("p", "."))

    if sigma_match:
        result["sigma"] = float(sigma_match.group(1).replace("p", "."))

    return result

path = "/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/TCGA_CompassNMD/Results/Histokit_30_06_2026/grid_search/blending_mode_constant__overlap_0p5/artifact_detection/grandqc/masks_cropped_color"

params = parse_grid_search_params(path)
print(params)

{'mode_overlap': 'constant', 'overlap': 0.5, 'sigma': ''}


In [4]:
main_dir = "/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/TCGA_CompassNMD/Results/Histokit_30_06_2026/grid_search"
folders_processed = os.listdir(main_dir)
folders_processed = [os.path.join(main_dir,f) for f in folders_processed if os.path.isdir(os.path.join(main_dir,f))]

In [5]:

target_mag = 10
for folder in  tqdm(folders_processed, desc="Processing folders"):

    mask_dir = os.path.join(folder, "artifact_detection/grandqc/masks")
    saver = HDF5Saver()
    parsed_dir = os.path.join(folder, "artifact_detection/grandqc/masks_cropped_numeric")

    parsed_color_dir = os.path.join(folder, "artifact_detection/grandqc/masks_cropped_color")

    os.makedirs(parsed_color_dir, exist_ok=True)

    os.makedirs(parsed_dir, exist_ok=True)

    #print("Processing folder:", folder)
    for m in os.listdir(mask_dir):
            name = m.split(".h5")[0]
            rows = df[df["svs_name"] == name]

            for row in rows.itertuples(index=False):
                try:
                    save_name = row.patch_path.split("/")[-1].split(".tiff")[0]
                    out_path = os.path.join( parsed_color_dir, f"{save_name}.png")

                    if os.path.exists(out_path):
                        try:
                            with Image.open(out_path) as img:
                                img.verify()

                            with Image.open(out_path) as img:
                                img.load()

                            # print(f"Skipping existing valid file: {save_name}.png")
                            continue

                        except Exception as e:
                            print(f"Existing file is corrupted, regenerating: {out_path}")
                            print(e)

                    print(f"Processing {folder} file: {save_name}.png")


                    x10 = row.x_10x
                    y10 = row.y_10x
                    w10 = row.patch_width_10x
                    h10 = row.patch_height_10x

                    dict_mask = saver.load(os.path.join(mask_dir, m))
                    size = dict_mask['level_dimensions_0']
                    mag = dict_mask['mag_l0']
                    factor = target_mag / mag
                    size = np.round(size * factor).astype(int)
                    slide = np.zeros((size[1], size[0]), dtype=np.uint8)

                    for b, mask in zip(dict_mask["bbox"], dict_mask["mask"]):
                        x, y, w, h = [float(v) for v in b]

                        x0 = int(np.floor(x))
                        y0 = int(np.floor(y))
                        x1 = int(np.ceil(x + w))
                        y1 = int(np.ceil(y + h))

                        mask = mask.astype(np.uint8)

                        x0_clip = max(0, x0)
                        y0_clip = max(0, y0)
                        x1_clip = min(x1, slide.shape[1])
                        y1_clip = min(y1, slide.shape[0])

                        if x1_clip <= x0_clip or y1_clip <= y0_clip:
                            continue

                        roi = slide[y0_clip:y1_clip, x0_clip:x1_clip]

                        mask_x0 = x0_clip - x0
                        mask_y0 = y0_clip - y0

                        roi_h, roi_w = roi.shape[:2]

                        mask_crop = mask[
                            mask_y0:mask_y0 + roi_h,
                            mask_x0:mask_x0 + roi_w
                        ]

                        common_h = min(roi.shape[0], mask_crop.shape[0])
                        common_w = min(roi.shape[1], mask_crop.shape[1])

                        roi = roi[:common_h, :common_w]
                        mask_crop = mask_crop[:common_h, :common_w]

                        con = (roi == 0) & (mask_crop != 0)
                        roi[con] = mask_crop[con]

                        slide[
                            y0_clip:y0_clip + common_h,
                            x0_clip:x0_clip + common_w
                        ] = roi

                    slide = slide[round(int(y10)):round(int(y10+h10)), int(round(x10)):int(round(x10+w10))]
                    Image.fromarray(slide).save(os.path.join(parsed_dir, f"{save_name}.png"))

                    slide_color = np.zeros((slide.shape[0], slide.shape[1], 3), dtype=np.uint8)
                    slide_color[slide == 0] = [0, 0, 0]
                    slide_color[slide == 1] = [128, 128, 128]
                    slide_color[slide == 2] = [255, 99, 71]
                    slide_color[slide == 3] = [0, 255, 0]
                    slide_color[slide == 4] = [255, 0, 0]
                    slide_color[slide == 5] = [255, 0, 255]
                    slide_color[slide == 6] = [75, 0, 130]

                    Image.fromarray(slide_color).save(os.path.join(parsed_color_dir, f"{save_name}.png"))
                except Exception as e:
                    print(f"Error processing {m}")
                    print(e)




Processing folders:  80%|████████  | 16/20 [02:52<00:43, 10.92s/it]

Processing /mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/TCGA_CompassNMD/Results/Histokit_30_06_2026/grid_search/blending_mode_gaussian__overlap_0p75__blending_sigma_1p25 file: TCGA-EC-A1QX-01Z-00-DX1.D94B9C3B-F06F-4300-B492-E2534AF4D703_R1.png


Processing folders: 100%|██████████| 20/20 [04:42<00:00, 14.12s/it]


Sometimes we have overlaping tissue regions, so we need to mask this areas, to do that we used masks defined for ground truth.

In [6]:
masks_dir = "/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/TCGA_CompassNMD/10x/annotated_masked_overlap/"
masks_names = os.listdir(masks_dir)

for folder in tqdm(folders_processed, desc="Processing folders"):
    parsed_color_dir = os.path.join(folder, "artifact_detection/grandqc/masks_cropped_color")

    for m_n in tqdm(masks_names, desc="Processing masks", leave=False):
        try:
            mask_path = os.path.join(masks_dir, m_n)
            pred_path = os.path.join(parsed_color_dir, m_n)

            mask = np.array(Image.open(mask_path).convert("L"))
            pred_mask = np.array(Image.open(pred_path).convert("RGB"))

            if mask.shape != pred_mask.shape[:2]:
                print(f"Shape mismatch for {m_n} in {folder}: mask {mask.shape}, pred {pred_mask.shape}")
                continue

            pred_mask[mask == 255] = [0, 0, 0]

            Image.fromarray(pred_mask).save(pred_path)

        except Exception as e:
            print(f"Error processing {m_n} in folder {folder}: {e}")





Processing folders: 100%|██████████| 20/20 [01:59<00:00,  5.97s/it]


Create prediction masks - visualise segmentation results for multiclass classification and calculate statistics.

In [7]:
from skimage.morphology import binary_opening, binary_closing, disk
from scipy.ndimage import binary_fill_holes
from skimage.morphology import remove_small_holes
from PIL.ImageFile import ImageFile
from concurrent.futures import ThreadPoolExecutor, as_completed
from histokit.segmentation.evaluate.eval import evaluate_rgb_mask
ImageFile.LOAD_TRUNCATED_IMAGES = True
from skimage.morphology import remove_small_holes
import skimage
from PIL.ImageFile import ImageFile
from concurrent.futures import ThreadPoolExecutor, as_completed
from histokit.segmentation.evaluate.eval import evaluate_rgb_mask

gt_folder = "/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/TCGA_CompassNMD/10x/gt_mask"
masks = os.listdir(gt_folder)
res_binary_list = []
res_multiclass_list = []
errors = []
tasks = []

def process_single_mask(folder, mask, gt_folder, classes):
    masks_color = os.path.join(folder, "artifact_detection/grandqc/masks_cropped_color")
    vis = os.path.join(folder, "artifact_detection/grandqc/visualization_segmentation")

    os.makedirs(vis, exist_ok=True)

    params = parse_grid_search_params(folder)

    mask_basename = os.path.basename(mask)

    gt_path = os.path.join(gt_folder, mask)
    pred_path = os.path.join(masks_color, mask_basename)

    if not os.path.exists(pred_path):
        return None, None, f"Missing prediction for {mask_basename}"

    mask_gt = np.array(Image.open(gt_path).convert("RGB"))
    mask_pred = np.array(Image.open(pred_path).convert("RGB"))

    if mask_gt.shape != mask_pred.shape:
        return None, None, (
            f"Shape mismatch for {mask_basename}: "
            f"GT {mask_gt.shape}, pred {mask_pred.shape}"
        )

    res_binary, res_multiclass = evaluate_rgb_mask(
        mask_gt=mask_gt,
        mask_pred=mask_pred,
        mask_basename=mask_basename,
        vis_dir=vis,
        method="HistoKit (no postprocessing)",
        tissue_class=[128, 128, 128],
        bg_class=[0, 0, 0],
        multiclass=True,
        class_dict=classes,
    )

    res_binary["Mode"] = params["mode_overlap"]
    res_binary["Overlap"] = params["overlap"]
    res_binary["Sigma"] = params["sigma"]
    res_multiclass["Mode"] = params["mode_overlap"]
    res_multiclass["Overlap"] = params["overlap"]
    res_multiclass["Sigma"] = params["sigma"]


    return res_binary, res_multiclass, None


res_binary_list = []
res_multiclass_list = []
errors = []
tasks = []

with ThreadPoolExecutor(max_workers=12) as executor:
    for folder in folders_processed:
        for mask in masks:
            tasks.append(
                executor.submit(
                    process_single_mask,
                    folder,
                    mask,
                    gt_folder,
                    classes,
                )
            )

    for future in tqdm(as_completed(tasks), total=len(tasks), desc="Processing masks"):
        try:
            res_binary, res_multiclass, error = future.result()

            if error is not None:
                errors.append(error)
                print(error)
                continue

            if res_binary is not None:
                res_binary_list.append(res_binary)

            if res_multiclass is not None:
                res_multiclass_list.append(res_multiclass)

        except Exception as e:
            errors.append(str(e))
            print(e)

df_binary = pd.DataFrame(res_binary_list)
df_multiclass = pd.DataFrame(res_multiclass_list)
df_errors = pd.DataFrame({"error": errors})

df_binary.to_csv("/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/TCGA_CompassNMD/Results/Histokit_30_06_2026/binary_metrics.csv", index=False)
df_multiclass.to_csv("/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/TCGA_CompassNMD/Results/Histokit_30_06_2026/multiclass_metrics.csv", index=False)
df_errors.to_csv("/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/TCGA_CompassNMD/Results/Histokit_30_06_2026/errors.csv", index=False)

Processing masks: 100%|██████████| 500/500 [24:23<00:00,  2.93s/it]


In [ ]:
from skimage.morphology import binary_opening, binary_closing, disk
from scipy.ndimage import binary_fill_holes
from PIL.ImageFile import ImageFile
from concurrent.futures import ThreadPoolExecutor, as_completed
from histokit.segmentation.evaluate.eval import evaluate_rgb_mask
ImageFile.LOAD_TRUNCATED_IMAGES = True
from skimage.morphology import remove_small_holes
import skimage
from PIL.ImageFile import ImageFile
from concurrent.futures import ThreadPoolExecutor, as_completed
from histokit.segmentation.evaluate.eval import evaluate_rgb_mask

gt_folder = "/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/TCGA_CompassNMD/10x/gt_mask"
masks = os.listdir(gt_folder)
res_binary_list = []
res_multiclass_list = []
errors = []
tasks = []

def process_single_mask(folder, mask, gt_folder, classes):
    masks_color = os.path.join(folder, "artifact_detection/grandqc/masks_cropped_color")
    vis = os.path.join(folder, "artifact_detection/grandqc/visualization_segmentation")
    processed = os.path.join(folder, "artifact_detection/grandqc/masks_cropped_color_postprocessed")
    os.makedirs(vis, exist_ok=True)
    os.makedirs(processed, exist_ok=True)
    params = parse_grid_search_params(folder)

    mask_basename = os.path.basename(mask)

    gt_path = os.path.join(gt_folder, mask)
    pred_path = os.path.join(masks_color, mask_basename)

    if not os.path.exists(pred_path):
        return None, None, f"Missing prediction for {mask_basename}"

    mask_gt = np.array(Image.open(gt_path).convert("RGB"))
    mask_pred = np.array(Image.open(pred_path).convert("RGB"))

    selem = disk(3)
    edge = np.all(mask_pred == classes["Edge"], axis=-1)
    edge = skimage.morphology.opening(edge, footprint=selem)
    edge = skimage.morphology.closing(edge, footprint=selem)
    edge = binary_fill_holes(edge)
    mask_pred[edge] = classes["Edge"]

    tissue = np.all(mask_pred == classes["Tissue"], axis=-1)

    tissue_filled = remove_small_holes(
        tissue,
        max_size=int(0.001 * tissue.shape[0] * tissue.shape[1])
    )

    holes = tissue_filled & ~tissue
    mask_pred[holes] = classes["Tissue"]

    oof = np.all(mask_pred == classes["Out.Of.Focus"], axis=-1)
    bg = np.all(mask_pred == classes["Background"], axis=-1)

    oof_processed = skimage.morphology.opening(oof, footprint=selem)
    oof_processed = skimage.morphology.closing(oof_processed, footprint=selem)
    oof_processed = oof_processed & ~bg
    oof_processed = remove_small_holes(
        oof_processed,
        max_size=int(0.001 * tissue.shape[0] * tissue.shape[1])
    )

    mask_pred[oof_processed] = classes["Out.Of.Focus"]

    Image.fromarray(mask_pred).save(os.path.join(processed, mask_basename))

    if mask_gt.shape != mask_pred.shape:
        return None, None, (
            f"Shape mismatch for {mask_basename}: "
            f"GT {mask_gt.shape}, pred {mask_pred.shape}"
        )

    res_binary, res_multiclass = evaluate_rgb_mask(
        mask_gt=mask_gt,
        mask_pred=mask_pred,
        mask_basename=mask_basename,
        vis_dir=vis,
        method="HistoKit (postprocessed)",
        tissue_class=[128, 128, 128],
        bg_class=[0, 0, 0],
        multiclass=True,
        class_dict=classes,
    )

    res_binary["Mode"] = params["mode_overlap"]
    res_binary["Overlap"] = params["overlap"]
    res_binary["Sigma"] = params["sigma"]
    res_multiclass["Mode"] = params["mode_overlap"]
    res_multiclass["Overlap"] = params["overlap"]
    res_multiclass["Sigma"] = params["sigma"]


    return res_binary, res_multiclass, None


res_binary_list = []
res_multiclass_list = []
errors = []
tasks = []

with ThreadPoolExecutor(max_workers=12) as executor:
    for folder in folders_processed:
        for mask in masks:
            tasks.append(
                executor.submit(
                    process_single_mask,
                    folder,
                    mask,
                    gt_folder,
                    classes,
                )
            )

    for future in tqdm(as_completed(tasks), total=len(tasks), desc="Processing masks"):
        try:
            res_binary, res_multiclass, error = future.result()

            if error is not None:
                errors.append(error)
                print(error)
                continue

            if res_binary is not None:
                res_binary_list.append(res_binary)

            if res_multiclass is not None:
                res_multiclass_list.append(res_multiclass)

        except Exception as e:
            errors.append(str(e))
            print(e)

df_binary = pd.DataFrame(res_binary_list)
df_multiclass = pd.DataFrame(res_multiclass_list)
df_errors = pd.DataFrame({"error": errors})

df_binary.to_csv("/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/TCGA_CompassNMD/Results/Histokit_30_06_2026/binary_metrics_processed.csv", index=False)
df_multiclass.to_csv("/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/TCGA_CompassNMD/Results/Histokit_30_06_2026/multiclass_metrics_processed.csv", index=False)
df_errors.to_csv("/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/TCGA_CompassNMD/Results/Histokit_30_06_2026/errors_processed.csv", index=False)



Processing masks:  18%|█▊        | 88/500 [05:29<29:03,  4.23s/it] 

## Slidl Dataset

This dataset contains information about two classes: artifact free tissue and regions containing artifacts or background. Artifacts like out of focus regions are not annotated, so there is a need to exclude this type of artifact from the masks.

In [4]:
main_dir = "/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/Slidl_dataset/Results/Histokit_30_06_2026/grid_search"
folders_processed = os.listdir(main_dir)
folders_processed = [os.path.join(main_dir,f) for f in folders_processed if os.path.isdir(os.path.join(main_dir,f))]


target_mag = 10
for folder in  tqdm(folders_processed, desc="Processing folders"):

    print("Processing folder:", folder)
    mask_dir = os.path.join(folder, "artifact_detection/grandqc/masks")
    saver = HDF5Saver()
    parsed_dir = os.path.join(folder, "artifact_detection/grandqc/masks_cropped_numeric")
    parsed_color_dir = os.path.join(folder, "artifact_detection/grandqc/masks_cropped_color")
    os.makedirs(parsed_color_dir, exist_ok=True)
    os.makedirs(parsed_dir, exist_ok=True)

    for m in os.listdir(mask_dir):
        try:
            name = m.split(".h5")[0]
            dict_mask = saver.load(os.path.join(mask_dir, m))
            size = dict_mask['level_dimensions_0']
            mag = dict_mask['mag_l0']
            factor = target_mag / mag
            size = np.round(size * factor).astype(int)
            slide = np.zeros((size[1], size[0]), dtype=np.uint8)

            for b, mask in zip(dict_mask["bbox"], dict_mask["mask"]):
                x, y, w, h = [float(v) for v in b]

                x0 = int(np.floor(x))
                y0 = int(np.floor(y))
                x1 = int(np.ceil(x + w))
                y1 = int(np.ceil(y + h))

                mask = mask.astype(np.uint8)

                x0_clip = max(0, x0)
                y0_clip = max(0, y0)
                x1_clip = min(x1, slide.shape[1])
                y1_clip = min(y1, slide.shape[0])

                if x1_clip <= x0_clip or y1_clip <= y0_clip:
                    continue

                roi = slide[y0_clip:y1_clip, x0_clip:x1_clip]

                mask_x0 = x0_clip - x0
                mask_y0 = y0_clip - y0

                roi_h, roi_w = roi.shape[:2]

                mask_crop = mask[
                    mask_y0:mask_y0 + roi_h,
                    mask_x0:mask_x0 + roi_w
                ]

                common_h = min(roi.shape[0], mask_crop.shape[0])
                common_w = min(roi.shape[1], mask_crop.shape[1])

                roi = roi[:common_h, :common_w]
                mask_crop = mask_crop[:common_h, :common_w]

                con = (roi == 0) & (mask_crop != 0)
                roi[con] = mask_crop[con]

                slide[
                    y0_clip:y0_clip + common_h,
                    x0_clip:x0_clip + common_w
                ] = roi

            Image.fromarray(slide).save(os.path.join(parsed_dir, f"{name}.png"))

            slide_color = np.zeros((slide.shape[0], slide.shape[1], 3), dtype=np.uint8)
            slide_color[slide == 0] = [0, 0, 0]
            slide_color[slide == 1] = [128, 128, 128]
            slide_color[slide == 2] = [255, 99, 71]
            slide_color[slide == 3] = [0, 255, 0]
            slide_color[slide == 4] = [255, 0, 0]
            slide_color[slide == 5] = [255, 0, 255]
            slide_color[slide == 6] = [75, 0, 130]

            Image.fromarray(slide_color).save(os.path.join(parsed_color_dir, f"{name}.png"))
        except Exception as e:
            print(f"Error processing {m}")
            print(e)

Processing folders:   0%|          | 0/20 [00:00<?, ?it/s]

Processing folder: /mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/Slidl_dataset/Results/Histokit_30_06_2026/grid_search/blending_mode_gaussian__overlap_0p25__blending_sigma_1p75


Processing folders:   5%|▌         | 1/20 [11:45<3:43:15, 705.03s/it]

Processing folder: /mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/Slidl_dataset/Results/Histokit_30_06_2026/grid_search/blending_mode_gaussian__overlap_0p5__blending_sigma_1p75


Processing folders:  10%|█         | 2/20 [23:20<3:29:53, 699.66s/it]

...

Processing folders:  90%|█████████ | 18/20 [3:21:36<24:10, 725.02s/it]

Processing folder: /mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/Slidl_dataset/Results/Histokit_30_06_2026/grid_search/blending_mode_constant__overlap_0p25


Processing folders:  95%|█████████▌| 19/20 [3:31:24<11:24, 684.09s/it]

Processing folder: /mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/Slidl_dataset/Results/Histokit_30_06_2026/grid_search/blending_mode_gaussian__overlap_0p75__blending_sigma_1p75


Processing folders: 100%|██████████| 20/20 [3:41:54<00:00, 665.73s/it]


Check if all files exists both in gt and prediction folders.

In [4]:
grid_folder = f"/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/Slidl_dataset/Results/Histokit_30_06_2026/grid_search/"
for pred_folder_main in tqdm(os.listdir(grid_folder), "Checking folders"):
    try:
        if not os.path.isdir(os.path.join(grid_folder, pred_folder_main)):
            continue
    except Exception as e:
        print(f"Error checking folder {pred_folder_main}")
        continue

    pred_folder = os.path.join(grid_folder, pred_folder_main, "artifact_detection/grandqc/masks_cropped_color")
    gt_dir = f"/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/Slidl_dataset/gt_masks"
    res = check_gt_pred_folders(gt_dir, pred_folder, use_ext = False)

    if not res:
        print(f"Mismatch in number of images in folder {pred_folder}")

Checking folders: 100%|██████████| 20/20 [00:00<00:00, 695.19it/s]


In [3]:
from skimage.morphology import remove_small_holes
import skimage
from PIL.ImageFile import ImageFile
from concurrent.futures import ThreadPoolExecutor, as_completed
from histokit.segmentation.evaluate.eval import evaluate_rgb_mask
from skimage.morphology import binary_opening, binary_closing, disk
from scipy.ndimage import binary_fill_holes
from PIL.ImageFile import ImageFile
from concurrent.futures import ThreadPoolExecutor, as_completed
from histokit.segmentation.evaluate.eval import evaluate_rgb_mask
ImageFile.LOAD_TRUNCATED_IMAGES = True
from skimage.morphology import remove_small_holes
import skimage
from PIL.ImageFile import ImageFile
from concurrent.futures import ThreadPoolExecutor, as_completed
from histokit.segmentation.evaluate.eval import evaluate_rgb_mask


main_dir = "/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/Slidl_dataset/Results/Histokit_30_06_2026/grid_search"
folders_processed = os.listdir(main_dir)
folders_processed = [os.path.join(main_dir,f) for f in folders_processed if os.path.isdir(os.path.join(main_dir,f))]

gt_folder = "/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/Slidl_dataset/gt_masks"
masks = os.listdir(gt_folder)
res_binary_list = []
res_multiclass_list = []
errors = []
tasks = []

classes = {
    "Tissue": [128, 128, 128],
    "Background": [0, 0, 0],
    "Fold": [255, 99, 71],
    "Dark.Spot": [0, 255, 0],
    "Pen": [255, 0, 0],
    "Edge": [255, 0, 255],
    "Out.Of.Focus": [75, 0, 130],
}


def process_single_mask(folder, mask, gt_folder, classes):
    masks_color = os.path.join(
        folder,
        "artifact_detection/grandqc/masks_cropped_color",
    )
    vis = os.path.join(
        folder,
        "artifact_detection/grandqc/visualization_segmentation",
    )
    postprocessed = os.path.join(
        folder,
        "artifact_detection/grandqc/masks_cropped_color_postprocessed",
    )

    os.makedirs(vis, exist_ok=True)
    os.makedirs(postprocessed, exist_ok=True)

    params = parse_grid_search_params(folder)

    mask_basename = os.path.basename(mask)
    gt_path = os.path.join(gt_folder, mask_basename)
    pred_path = os.path.join(masks_color, mask_basename)

    if not os.path.exists(gt_path):
        return None, None, f"Missing ground truth for {mask_basename}"

    if not os.path.exists(pred_path):
        return None, None, f"Missing prediction for {mask_basename}"

    # Ground truth
    mask_gt_rgb = np.array(Image.open(gt_path).convert("RGB"))

    # GT: white = tissue, everything else = background
    gt_tissue = np.all(mask_gt_rgb == [255, 255, 255], axis=-1)

    mask_gt_bin = np.zeros(gt_tissue.shape, dtype=np.uint8)
    mask_gt_bin[gt_tissue] = 255

    # Prediction
    mask_pred_rgb = np.array(Image.open(pred_path).convert("RGB"))

    # Resize prediction before processing if necessary
    if mask_pred_rgb.shape[:2] != mask_gt_rgb.shape[:2]:
        print(
            f"Resizing {mask_basename}: "
            f"pred {(mask_pred_rgb.shape[1], mask_pred_rgb.shape[0])} "
            f"-> GT {(mask_gt_rgb.shape[1], mask_gt_rgb.shape[0])}"
        )

        mask_pred_rgb = np.array(
            Image.fromarray(mask_pred_rgb).resize(
                (mask_gt_rgb.shape[1], mask_gt_rgb.shape[0]),
                resample=Image.Resampling.NEAREST,
            )
        )

    selem = disk(3)
    edge = np.all(mask_pred_rgb == classes["Edge"], axis=-1)
    edge = skimage.morphology.opening(edge, footprint=selem)
    edge = skimage.morphology.closing(edge, footprint=selem)
    edge = binary_fill_holes(edge)
    mask_pred_rgb[edge] = classes["Edge"]

    tissue = np.all(mask_pred_rgb == classes["Tissue"], axis=-1)

    tissue_filled = remove_small_holes(
        tissue,
        max_size=int(0.001 * tissue.shape[0] * tissue.shape[1])
    )

    holes = tissue_filled & ~tissue
    mask_pred_rgb[holes] = classes["Tissue"]

    # Prediction: tissue + Out.Of.Focus = tissue (no annotation in this dataset)
    pred_tissue = (
        np.all(mask_pred_rgb == [128, 128, 128], axis=-1)
        | np.all(mask_pred_rgb == [75, 0, 130], axis=-1)
    )

    mask_pred_bin = np.zeros(pred_tissue.shape, dtype=np.uint8)
    mask_pred_bin[pred_tissue] = 255

    Image.fromarray(mask_pred_bin).save(
        os.path.join(postprocessed, mask_basename)
    )

    # Convert binary masks to RGB for evaluation
    mask_gt_eval = np.stack([mask_gt_bin] * 3, axis=-1)
    mask_pred_eval = np.stack([mask_pred_bin] * 3, axis=-1)

    res_binary, res_multiclass = evaluate_rgb_mask(
        mask_gt=mask_gt_eval,
        mask_pred=mask_pred_eval,
        mask_basename=mask_basename,
        vis_dir=vis,
        method="HistoKit (no postprocessing)",
        tissue_class=[255, 255, 255],
        bg_class=[0, 0, 0],
        multiclass=True,
        class_dict=classes,
    )

    for result in (res_binary, res_multiclass):
        result["Mode"] = params["mode_overlap"]
        result["Overlap"] = params["overlap"]
        result["Sigma"] = params["sigma"]

    return res_binary, res_multiclass, None


res_binary_list = []
res_multiclass_list = []
errors = []
tasks = []

with ThreadPoolExecutor(max_workers=3) as executor:
    for folder in folders_processed:
        for mask in masks:
            tasks.append(
                executor.submit(
                    process_single_mask,
                    folder,
                    mask,
                    gt_folder,
                    classes,
                )
            )

    for future in tqdm(as_completed(tasks), total=len(tasks), desc="Processing masks"):
        try:
            res_binary, res_multiclass, error = future.result()

            if error is not None:
                errors.append(error)
                print(error)
                continue

            if res_binary is not None:
                res_binary_list.append(res_binary)

            if res_multiclass is not None:
                res_multiclass_list.append(res_multiclass)

        except Exception as e:
            errors.append(str(e))
            print(e)

df_binary = pd.DataFrame(res_binary_list)
df_multiclass = pd.DataFrame(res_multiclass_list)
df_errors = pd.DataFrame({"error": errors})

df_binary.to_csv("/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/Slidl_dataset/Results/Histokit_30_06_2026/processed_binary_metrics.csv", index=False)
df_multiclass.to_csv("/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/Slidl_dataset/Results/Histokit_30_06_2026/processed_multiclass_metrics.csv", index=False)
df_errors.to_csv("/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/Slidl_dataset/Results/Histokit_30_06_2026/errors.csv", index=False)

Processing masks:   0%|          | 1/260 [02:57<12:47:13, 177.74s/it]

Resizing normal_004.png: pred (23399, 53146) -> GT (24064, 54656)


Processing masks:   2%|▏         | 4/260 [09:37<11:39:38, 163.98s/it]

Resizing tumor_031.png: pred (23773, 53769) -> GT (24448, 55296)


Processing masks:   2%|▏         | 5/260 [12:16<11:28:57, 162.11s/it]

...

Processing masks:  97%|█████████▋| 251/260 [10:29:57<23:29, 156.61s/it]

Resizing tumor_031.png: pred (23773, 53769) -> GT (24448, 55296)


Processing masks:  97%|█████████▋| 252/260 [10:30:49<16:41, 125.23s/it]

Resizing tumor_021.png: pred (23773, 53644) -> GT (24448, 55168)


Processing masks: 100%|██████████| 260/260 [10:52:38<00:00, 150.61s/it]


## GrandQC Test Dataset

In this dataset, we exclude background prediction, due to the fact that patches contain tissue and annotated artifacts.

In [2]:
organs = ["Breast", "Kidney", "Colon", "Prostate"]
target_mag = 10
for o in organs:
    main_dir = f"/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/GrandQC Test Dataset/PreprocessedDataset/{o}/10x/Results/Histokit_30_06_2026/grid_search"
    dirs_img = os.listdir(main_dir)
    folders_processed = [os.path.join(main_dir,f) for f in dirs_img]

    for folder in  tqdm(folders_processed, desc="Processing folders"):

        print("Processing folder:", folder)
        mask_dir = os.path.join(folder, "artifact_detection/grandqc/masks")
        saver = HDF5Saver()
        parsed_dir = os.path.join(folder, "artifact_detection/grandqc/masks_cropped_numeric")
        parsed_color_dir = os.path.join(folder, "artifact_detection/grandqc/masks_cropped_color")
        os.makedirs(parsed_color_dir, exist_ok=True)
        os.makedirs(parsed_dir, exist_ok=True)

        for m in os.listdir(mask_dir):
            try:
                name = m.split(".h5")[0]
                dict_mask = saver.load(os.path.join(mask_dir, m))
                size = dict_mask['level_dimensions_0']
                mag = dict_mask['mag_l0']
                factor = target_mag / mag
                size = np.round(size * factor).astype(int)
                slide = np.zeros((size[1], size[0]), dtype=np.uint8)

                for b, mask in zip(dict_mask["bbox"], dict_mask["mask"]):
                    x, y, w, h = [float(v) for v in b]

                    x0 = int(np.floor(x))
                    y0 = int(np.floor(y))
                    x1 = int(np.ceil(x + w))
                    y1 = int(np.ceil(y + h))

                    mask = mask.astype(np.uint8)

                    x0_clip = max(0, x0)
                    y0_clip = max(0, y0)
                    x1_clip = min(x1, slide.shape[1])
                    y1_clip = min(y1, slide.shape[0])

                    if x1_clip <= x0_clip or y1_clip <= y0_clip:
                        continue

                    roi = slide[y0_clip:y1_clip, x0_clip:x1_clip]

                    mask_x0 = x0_clip - x0
                    mask_y0 = y0_clip - y0

                    roi_h, roi_w = roi.shape[:2]

                    mask_crop = mask[
                        mask_y0:mask_y0 + roi_h,
                        mask_x0:mask_x0 + roi_w
                    ]

                    common_h = min(roi.shape[0], mask_crop.shape[0])
                    common_w = min(roi.shape[1], mask_crop.shape[1])

                    roi = roi[:common_h, :common_w]
                    mask_crop = mask_crop[:common_h, :common_w]

                    con = (roi == 0) & (mask_crop != 0)
                    roi[con] = mask_crop[con]

                    slide[
                        y0_clip:y0_clip + common_h,
                        x0_clip:x0_clip + common_w
                    ] = roi

                Image.fromarray(slide).save(os.path.join(parsed_dir, f"{name}.png"))

                slide_color = np.zeros((slide.shape[0], slide.shape[1], 3), dtype=np.uint8)
                slide_color[slide == 0] = [0, 0, 0]
                slide_color[slide == 1] = [128, 128, 128]
                slide_color[slide == 2] = [255, 99, 71]
                slide_color[slide == 3] = [0, 255, 0]
                slide_color[slide == 4] = [255, 0, 0]
                slide_color[slide == 5] = [255, 0, 255]
                slide_color[slide == 6] = [75, 0, 130]

                Image.fromarray(slide_color).save(os.path.join(parsed_color_dir, f"{name}.png"))
            except Exception as e:
                print(f"Error processing {m}")
                print(e)


Processing folders:   0%|          | 0/20 [00:00<?, ?it/s]

Processing folder: /mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/GrandQC Test Dataset/PreprocessedDataset/Breast/10x/Results/Histokit_30_06_2026/grid_search/blending_mode_gaussian__overlap_0p25__blending_sigma_1p75


Processing folders:   5%|▌         | 1/20 [01:55<36:26, 115.07s/it]

Processing folder: /mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/GrandQC Test Dataset/PreprocessedDataset/Breast/10x/Results/Histokit_30_06_2026/grid_search/blending_mode_gaussian__overlap_0p5__blending_sigma_1p75


Processing folders:  10%|█         | 2/20 [04:24<40:33, 135.21s/it]

...

Processing folders:  90%|█████████ | 18/20 [1:16:31<07:12, 216.25s/it]

Processing folder: /mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/GrandQC Test Dataset/PreprocessedDataset/Prostate/10x/Results/Histokit_30_06_2026/grid_search/blending_mode_constant__overlap_0p25


Processing folders:  95%|█████████▌| 19/20 [1:21:15<03:56, 236.48s/it]

Processing folder: /mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/GrandQC Test Dataset/PreprocessedDataset/Prostate/10x/Results/Histokit_30_06_2026/grid_search/blending_mode_gaussian__overlap_0p75__blending_sigma_1p75


Processing folders: 100%|██████████| 20/20 [1:24:17<00:00, 252.90s/it]


In [3]:
## Check number of images in each folder (should be the same as the number of gt images)
organs = ["Kidney","Colon", "Breast", "Prostate"]

for o in organs:
    grid_folder = f"/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/GrandQC Test Dataset/PreprocessedDataset/{o}/10x/Results/Histokit_30_06_2026/grid_search"
    for pred_folder_main in tqdm(os.listdir(grid_folder), "Checking folders for organ: " + o):
        try:
            if not os.path.isdir(os.path.join(grid_folder, pred_folder_main)):
                continue
        except Exception as e:
            print(f"Error checking folder {pred_folder_main} for organ {o}: {e}")
            continue

        pred_folder = os.path.join(grid_folder, pred_folder_main, "artifact_detection/grandqc/masks_cropped_color")
        gt_dir = f"/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/GrandQC Test Dataset/PreprocessedDataset/{o}/10x/gt_mask"
        res = check_gt_pred_folders(gt_dir, pred_folder, use_ext = False)

        for pred_img_pth in os.listdir(pred_folder):
            pred_img = Image.open(os.path.join(pred_folder, pred_img_pth))
            gt_img = Image.open(os.path.join(gt_dir, pred_img_pth))
            if pred_img.size != gt_img.size:
                print(f"Size mismatch for {pred_img} in organ {o}: GT {gt_img.size}, pred {pred_img.size}")

        if not res:
            print(f"Mismatch in number of images for organ {o} in folder {pred_folder}")

Checking folders for organ: Prostate: 100%|██████████| 20/20 [00:32<00:00,  1.62s/it]


In [ ]:
from histokit.slide import Slide

s = Slide("/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/Slidl_dataset/svs/tumor_021.tif")